In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
from google.colab import drive
drive.mount('/content/drive')

!ls -lh /content/drive/MyDrive/*.zip

Mounted at /content/drive
-rw------- 1 root root  13M Sep 15 14:19 /content/drive/MyDrive/full_rgb_s16_256.zip
-rw------- 1 root root  13M Sep 12 14:00 /content/drive/MyDrive/full_rgb_s16.zip
-rw------- 1 root root 527M Sep 22 13:45 /content/drive/MyDrive/neg_256.zip
-rw------- 1 root root 566M Sep  7 20:27 /content/drive/MyDrive/tensors_messy_10ms_packed.zip
-rw------- 1 root root 1.1G Sep  7 20:28 /content/drive/MyDrive/tensors_messy_40ms_packed.zip
-rw------- 1 root root 580M Sep  7 16:39 /content/drive/MyDrive/tensors_messy_packed.zip
-rw------- 1 root root 1.5G Sep 15 11:30 /content/drive/MyDrive/tensors_rgb_256_packed_k.zip
-rw------- 1 root root 1.2G Sep  9 11:34 /content/drive/MyDrive/tensors_rgb_packed_k.zip
-rw------- 1 root root 1.2G Sep  9 11:22 /content/drive/MyDrive/tensors_rgb_packed.zip


In [3]:
DRONES = '/content/drive/MyDrive/tensors_rgb_256_packed_k.zip'
NEGS   = '/content/drive/MyDrive/neg_256.zip'

import os
for p in (DRONES, NEGS):
    assert os.path.exists(p), f'not found: {p}'
print('both archives found')

both archives found


In [4]:
ROOT = '/content/drone'
TENSORS = f'{ROOT}/Data_new/tensors_rgb/tensors_rgb_256_packed'

!mkdir -p {TENSORS} {ROOT}/Data_new/runs

!unzip -qo {DRONES} -d {TENSORS}
!unzip -qo {NEGS}   -d {TENSORS}

import glob, shutil, os
for sub in glob.glob(f'{TENSORS}/*/'):
    for f in glob.glob(sub + '*'):
        shutil.move(f, TENSORS)
    os.rmdir(sub)

n_npz = len(glob.glob(f'{TENSORS}/*_tensors.npz'))
print(f'clips: {n_npz}')
assert n_npz == 285, 'expected 285 clips (114 drone + 171 negative)'

Error: Destination path '/content/drone/Data_new/tensors_rgb/tensors_rgb_256_packed/splits.json' already exists

In [5]:
import glob, shutil, os

for sub in glob.glob(f'{TENSORS}/*/'):
    for f in glob.glob(sub + '*'):
        dst = os.path.join(TENSORS, os.path.basename(f))
        if os.path.exists(dst):
            os.remove(dst)          # novija kopija iz podfoldera pobjeđuje
        shutil.move(f, dst)
    os.rmdir(sub)

n_npz = len(glob.glob(f'{TENSORS}/*_tensors.npz'))
print(f'clips: {n_npz}')
assert n_npz == 285, 'expected 285 clips (114 drone + 171 negative)'

clips: 285


In [6]:
import json, shutil

shutil.copy(f'{TENSORS}/splits.json', f'{ROOT}/Data_new/splits.json')

splits = json.load(open(f'{ROOT}/Data_new/splits.json'))

for k in ('train', 'validation', 'test'):
    print(f'{k:<12}{len(splits[k]):>5} drone   '
          f'{len(splits.get(k + "_neg", [])):>5} negative')

assert splits.get('train_neg'), 'splits.json has no negatives -- wrong copy'

train          90 drone       0 negative
validation     12 drone       0 negative
test           12 drone       0 negative


AssertionError: splits.json has no negatives -- wrong copy

In [7]:
!git clone -q https://github.com/bornamuzina/akida-drone-detection /content/repo

import json, shutil
shutil.copy('/content/repo/Data_new/splits.json', f'{ROOT}/Data_new/splits.json')

splits = json.load(open(f'{ROOT}/Data_new/splits.json'))
for k in ('train', 'validation', 'test'):
    print(f'{k:<12}{len(splits[k]):>5} drone   '
          f'{len(splits.get(k + "_neg", [])):>5} negative')

assert splits.get('train_neg'), 'splits.json has no negatives'

train          90 drone     135 negative
validation     12 drone      17 negative
test           12 drone      19 negative


In [10]:
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
!apt-get -qq install -y python3.11 python3.11-dev > /dev/null 2>&1
!python3.11 --version

Python 3.11.15


In [11]:
!pip install -q virtualenv
!virtualenv -q -p python3.11 /content/akv
!/content/akv/bin/pip install -q -r /content/repo/requirements/requirements-akida.txt

!/content/akv/bin/python -c "import tensorflow as tf, tf_keras, akida; print('tf', tf.__version__, '| akida', akida.__version__, '| gpu', bool(tf.config.list_physical_devices('GPU')))"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
2026-09-22 14:05:22.640243: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790085922.660775    7327 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790085922.667624    7327 cuda_blas.cc:1407] Unable to register

In [12]:
!MPLBACKEND=Agg /content/akv/bin/python -c "import tensorflow as tf, tf_keras, akida; print('tf', tf.__version__, '| akida', akida.__version__, '| gpu', bool(tf.config.list_physical_devices('GPU')))"

2026-09-22 14:06:41.572878: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790086001.608840    7711 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790086001.620867    7711 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790086001.647291    7711 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790086001.647333    7711 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790086001.647340    7711 computation_placer.cc:177] computation placer alr

In [18]:
!nproc

2


In [19]:
%env DRONE_ROOT=/content/drone

!cd /content/repo/src/model && MPLBACKEND=Agg /content/akv/bin/python -u train.py \
    --epochs 10 \
    --negatives \
    --name yolov2_s16_rgb_256_neg

env: DRONE_ROOT=/content/drone
2026-09-22 14:45:06.385722: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790088306.422490   17674 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790088306.432525   17674 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790088306.459998   17674 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790088306.460051   17674 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790088306.460059   17674 computation_placer

In [20]:
RUN = 'yolov2_s16_rgb_256_neg'

!cd /content/repo/src/model && MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py \
    --run {RUN} --split test

2026-09-22 16:38:54.471879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790095134.517355   46528 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790095134.530859   46528 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790095134.576186   46528 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790095134.576264   46528 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790095134.576273   46528 computation_placer.cc:177] computation placer alr

In [22]:
!cd /content/repo/src/model && MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py \
    --run {RUN} --split validation --negatives

2026-09-22 16:40:48.524713: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790095248.546283   48851 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790095248.553283   48851 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790095248.569954   48851 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790095248.570003   48851 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790095248.570008   48851 computation_placer.cc:177] computation placer alr

In [25]:
!cd /content/repo/src/model && MPLBACKEND=Agg /content/akv/bin/python -u evaluate.py \
    --run yolov2_s16_rgb_256_neg --split test --negatives

2026-09-22 16:45:48.288933: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790095548.324666   51226 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790095548.336452   51226 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790095548.365538   51226 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790095548.365570   51226 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790095548.365578   51226 computation_placer.cc:177] computation placer alr

In [26]:
RUN = 'yolov2_s16_rgb_256_neg'

!cd /content/drone/Data_new/runs && zip -qr /content/{RUN}.zip {RUN}
!ls -lh /content/{RUN}.zip

from google.colab import files
files.download(f'/content/{RUN}.zip')

-rw-r--r-- 1 root root 26M Sep 22 17:16 /content/yolov2_s16_rgb_256_neg.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>